<a href="https://colab.research.google.com/github/GUNAPILLCO/neural_profit/blob/main/notebooks/stage_07_model_training/stage_07_00_model_training_seq2seq_plan.ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Stage_07 – Model Training (Seq2Seq) – Objetivo y Modelos**


## **1. Objetivo de predicción (definición formal)**


Entrenaremos modelos **seq2seq** para predecir una **secuencia futura completa** a partir de una ventana histórica intradía.

- **Entrada (X):** bloque de tamaño **29 × 8**  
  $$
  X \in \mathbb{R}^{29 \times 8}
  $$
  donde 29 = minutos consecutivos (ventana histórica) y 8 = features por minuto.

- **Salida / Target (Y):** bloque de tamaño **29 × 1**  
  $$
  Y \in \mathbb{R}^{29 \times 1}
  $$
  donde 29 = minutos futuros a predecir y 1 = variable objetivo escalar por minuto.

**Interpretación:** el modelo no predice un único valor final, sino la **trayectoria temporal futura completa** (29 pasos).





### **1.1. Qué se evalúa en este stage_07**

En el stage_07 NO se decide el “mejor modelo”.
Aquí solo se garantiza que:
- cada modelo fue entrenado correctamente,
- sin ver validation ni test,
- con complejidad controlada.

La comparación real viene recién en el siguinte stage_08

### **1.2. Qué NO es este problema**


Quedan explícitamente fuera de alcance:

- **Many-to-one:** predecir solo el último paso \(t+H\).
- **Targets agregados:** suma/promedio/retorno acumulado colapsado en un escalar.
- **Heads independientes por horizonte:** entrenar 29 regresiones separadas sin estructura temporal.

## **2. Entrenamiento de múltiples modelos**

Entrenaremos varios modelos, siempre bajo las mismas reglas:

- Mismo dataset.
- Mismo split temporal.
- Mismo escalamiento.
- Misma función objetivo.

Ejemplos típicos (alineados al libro):

- Naive (baseline)
- Modelos lineales (Ridge / Lasso)
- Árboles / ensembles
- Redes (MLP, LSTM, etc.)

Cada uno:
- se entrena solo con TRAIN,
- con regularización definida antes del ajuste.

## **3. Modelos a evaluar (comparación escalonada)**


Definiremos y entrenaremos un modelo base (baseline) para tener una referencia mínima.

Esto sirve para:
- detectar overfitting temprano,
- saber si un modelo “sofisticado” realmente aporta valor,
- evitar resultados engañosos.

Evaluaremos un set mínimo y robusto, desde baselines hasta modelos fuertes:

### **3.1. Modelos propuestos**

#### **(A) Baselines obligatorios**


1. **Naive / Persistence**
   - Predice “sin cambio” (por ejemplo, replica el último valor observado) en los 29 pasos.
   - Define el “piso” mínimo de performance.
   - Notebook: `stage_07_01_naive_model.ipynb`

2. **MLP (Direct Multi-step)**
   - Aplana $(29 \times 8)$ y predice $(29 \times 1)$.
   - Base neural simple y rápida para validar el pipeline.
   - Notebook: `stage_07_02_mlp_direct_multistep.ipynb`

#### **(B) Seq2Seq clásicos**


3. **Encoder–Decoder GRU**
    - Notebook: `stage_07_03_gru_seq2seq.ipynb`

4. **Encoder–Decoder LSTM**
    - Encoder resume la historia; decoder genera la secuencia futura.
    - Estables y comparables para intradía.
    - Notebook: `stage_07_04_lstm_seq2seq.ipynb`



#### **(C) Alternativa robusta no recurrente**


5. **TCN (Temporal Convolutional Network)**
   - Convoluciones causales dilatadas.
   - Buena relación performance/estabilidad.
   - Notebook: `stage_07_05_tcn_seq2seq.ipynb`

#### **(D) Modelo de atención**


6. **Transformer Seq2Seq**
   - Encoder–decoder con atención.
   - Requiere regularización y control cuidadoso, pero es candidato fuerte.
   - Notebook: `stage_07_06_transformer_seq2seq.ipynb`

#### **(E) Modelo avanzado con estructura temporal explícita**


7. **Temporal Fusion Transformer (TFT)**

- Modelo seq2seq con atención para series temporales multivariadas.
- Incorpora selección de variables y atención temporal para capturar dependencias pasadas y futuras.
- Alta capacidad para modelar patrones intradía complejos; referencia avanzada frente a LSTM, TCN y Transformer estándar.
- Notebook: `stage_07_07_tft_seq2seq.ipynb`

### **3.2. Regularización de modelos**

#### **(A) Baselines obligatorios**


1. **Naive / Persistence**

    Regularización:
    No aplica.

    - No tiene parámetros entrenables.
    - No puede sobreajustar en sentido de Machine Learning.

    Rol:
    Define el piso mínimo de performance.
    Si un modelo entrenable no supera este baseline en validation, se descarta.

2. **MLP (Direct Multi-step)**

    Riesgo principal:
    Sobreajuste por capacidad del modelo.

    Regularización recomendada:

    - Penalización L2 (weight decay) como mecanismo principal.
    - Early stopping monitoreando la pérdida en validation.
    - Arquitectura limitada:
      - pocas capas (1–2),
      - número reducido de neuronas.
    - Dropout leve (opcional, solo si se observa sobreajuste claro).

    Interpretación según el libro:
    Modelo flexible que requiere regularización explícita para generalizar.

#### **(B) Seq2Seq clásicos**


3. **Encoder–Decoder GRU**

    Riesgo:
    Memorización de secuencias específicas del conjunto de entrenamiento.

    Regularización recomendada:

    - Early stopping como mecanismo principal.
    - Limitar la dimensión del hidden state.
    - Limitar el número de capas (1–2).
    - Dropout opcional entre capas, no dentro de la recurrencia.

    Nota:
    GRU es más estable que LSTM y suele requerir menor regularización.

4. **Encoder–Decoder LSTM**

    Riesgo:
    Alta capacidad combinada con memoria de largo plazo.

    Regularización recomendada:

    - Early stopping obligatorio.
    - Dropout en entradas y salidas (no recurrente).
    - Tamaño moderado del hidden state.
    - Número de capas limitado.
    - Penalización L2 suave en los pesos (opcional).

    Interpretación según el libro:
    Modelo potente que requiere regularización estructural y temporal.


#### **(C) Alternativa robusta no recurrente**


5. **TCN (Temporal Convolutional Network)**

    Riesgo:
    Campo receptivo excesivo o filtros redundantes.

    Regularización recomendada:

    - Limitar el número de filtros por capa.
    - Limitar la profundidad del modelo (dilataciones).
    - Dropout (especialmente efectivo en TCN).
    - Weight normalization si está disponible.
    - Early stopping.

    Ventaja clave:
    Suele generalizar mejor que modelos recurrentes con menor regularización agresiva.

#### **(D) Modelo de atención**


6. **Transformer Seq2Seq**

    Riesgo:
    Sobreajuste severo debido a alta capacidad.

    Regularización obligatoria:

    - Dropout alto en bloques de atención y feed-forward.
    - Early stopping estricto.
    - Limitar el número de capas.
    - Limitar la dimensión del embedding.
    - Label smoothing opcional en esquemas de pérdida multi-step.

    Regla práctica:
    Sin regularización fuerte, el modelo no generaliza.

#### **(E) Modelo avanzado con estructura temporal explícita**


7. **Temporal Fusion Transformer (TFT)**

    Riesgo:
    Alta capacidad, parcialmente mitigada por su diseño estructurado.

    Regularización integrada en la arquitectura:

    - Redes de selección de variables.
    - Mecanismos de gating.
    - Dropout configurable.
    - Early stopping.

    Aspectos que deben controlarse explícitamente:

    - Dimensión del hidden state.
    - Número de capas LSTM internas.
    - Nivel de dropout global.

    Interpretación según el libro:
    Modelo avanzado que regulariza principalmente por arquitectura, no solo por penalización.

#### **Resumen operativo**


- Naive: sin regularización.
- MLP: L2 + early stopping.
- GRU / LSTM: early stopping + tamaño controlado + dropout.
- TCN: control de profundidad + dropout.
- Transformer: dropout fuerte + límites estrictos.
- TFT: regularización arquitectónica + early stopping.

La regularización no es un agregado opcional.
Forma parte del diseño del modelo y determina su capacidad de generalización.

Cada modelo requiere un esquema de regularización distinto.
La comparación real entre ellos se realiza recién en el siguiente stage_08

## **4. Definición de la comparación Predicción vs Valor Real (Seq2Seq)**


Para cada muestra del dataset, la evaluación del modelo se define de la siguiente manera:

### **4.1 Esquema de predicción**

- El modelo recibe un **bloque de entrada**:

  $$
  X_t \in \mathbb{R}^{29 \times 8}
  $$

  correspondiente a 29 minutos históricos y 8 features por minuto.

- A partir de ese bloque, el modelo **predice una secuencia futura completa**:

  $$
  \hat{Y}_{t+1:t+29} \in \mathbb{R}^{29 \times 1}
  $$

  es decir, un valor del target por cada uno de los 29 minutos futuros.




### **4.2 Comparación contra el valor real**


La predicción se compara directamente contra la **secuencia real futura observada**:

$$
Y_{t+1:t+29}^{real} \in \mathbb{R}^{29 \times 1}
$$

La comparación es siempre **secuencia contra secuencia**, sin colapsar el target ni aplicar agregaciones previas.




### **4.3 Cálculo de métricas**

A partir de esta comparación base, las métricas se calculan:

- **Por paso temporal**:  
  comparación entre $\hat{y}_{t+k}$ y $y_{t+k}^{real}$, para $(k = 1,\dots,29)$

- **Sobre la trayectoria completa**:  
  error global entre $\hat{Y}_{t+1:t+29}$ y $Y_{t+1:t+29}^{real}$.

No se compara contra un escalar ni contra un valor agregado final.  
El problema es estrictamente **seq2seq**.

### **4.4 Derivación de métricas**


Janse dice esto:

El modelo se entrena solo con TRAIN,
pero se evalúa con VALID para decidir si sirve o no.

Es decir:
- TRAIN → para aprender
- VALID → para medir desempeño y comparar modelos
- TEST → solo al final, una vez elegido el mejor modelo


Desde esta comparación fundamental se derivan:

- **Métricas de machine learning**: MAE, RMSE, métricas direccionales.
- **Métricas económicas**: EV, TP/SL, drawdown, usando el **delta real observado** dentro de la ventana futura.

## **5. Métricas de predicción (Machine Learning)**

Dado que el problema consiste en la predicción de una secuencia futura de 29 pasos, la evaluación del desempeño del modelo se realiza mediante un conjunto reducido de métricas finales, priorizando comparabilidad, estabilidad y alineación con el objetivo del proyecto.

Las métricas se calculan siempre fuera de muestra (validation), y el conjunto test se reserva exclusivamente para la evaluación final.


### **5.1 Métricas principales de error (seq2seq)**

La secuencia predicha $\hat{Y}_{t+1:t+29}$ se compara directamente contra la secuencia real $Y_{t+1:t+29}$.

Para cada modelo se reportan métricas globales, calculadas concatenando los 29 pasos de la secuencia en un único vector:

- MAE (Mean Absolute Error)  
  Error absoluto medio calculado sobre todos los pasos de la secuencia.

- RMSE (Root Mean Squared Error)  
  Raíz del error cuadrático medio calculada sobre todos los pasos de la secuencia.

Estas métricas constituyen el criterio principal para la comparación y ranking de modelos, de acuerdo con el enfoque del libro.


### **5.2 Análisis del error por horizonte (diagnóstico)**


Con fines de análisis interno, el error puede descomponerse por paso temporal:

- MAE(k), para k = 1, …, 29

Este análisis permite:
- observar la degradación del error a medida que aumenta el horizonte,
- analizar la estabilidad temporal de cada modelo.

Este análisis es estrictamente diagnóstico y no se utiliza para el ranking ni la
selección final de modelos.


### **5.3 Métrica direccional (complementaria)**


Como métrica complementaria, se utiliza la Directional Accuracy (DA), definida como la proporción de casos en los que el signo del delta predicho coincide con el signo del delta real.

Para mantener una definición única y consistente, se utiliza:

- DA_last  
  Coincidencia de signo en el último paso de la secuencia (k = 29).

Esta métrica conecta directamente con la dirección del movimiento esperado al
horizonte H y se utiliza únicamente con fines interpretativos, no como criterio
principal de selección.




### **5.4 Coeficiente de determinación**


Se reporta adicionalmente el coeficiente de determinación R², calculado sobre la
secuencia completa.

Esta métrica se considera complementaria y descriptiva, y no se utiliza como
criterio principal de comparación debido a su limitada estabilidad en series
financieras.

### **Jerarquía de métricas**

- Métricas principales de comparación:
  MAE, RMSE (globales, seq2seq)

- Métricas complementarias:
  Directional Accuracy (DA_last), R²

- Métricas diagnósticas:
  Error por paso MAE(k)

La selección de modelos se realiza exclusivamente en base a las métricas principales, evaluadas sobre el conjunto de validation.

### 5.5. **Función de cálculo de métricas**

In [ ]:
import numpy as np
from sklearn.metrics import r2_score

def compute_seq2seq_metrics(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    compute_r2: bool = True,
    da_ignore_zeros: bool = True,
) -> dict:
    """
    Calcula métricas simples y comparables para modelos seq2seq.

    Parámetros
    ----------
    y_true : np.ndarray
        Valores reales con shape (n_samples, seq_len) o (n_samples, seq_len, 1)
    y_pred : np.ndarray
        Valores predichos con shape (n_samples, seq_len) o (n_samples, seq_len, 1)
    compute_r2 : bool
        Si True, calcula R² sobre la secuencia completa concatenada.
    da_ignore_zeros : bool
        Si True, ignora casos donde el signo sea 0 (y_true_last==0 o y_pred_last==0)
        al calcular DA_last. Esto evita ambigüedad en la dirección.

    Retorna
    -------
    metrics : dict
        Diccionario con métricas globales y diagnóstico por paso.
    """

    # 1) Convertir a np.ndarray y forzar float
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)

    # 2) Normalizar dimensiones a (n_samples, seq_len)
    if y_true.ndim == 3 and y_true.shape[-1] == 1:
        y_true = y_true.squeeze(-1)
    if y_pred.ndim == 3 and y_pred.shape[-1] == 1:
        y_pred = y_pred.squeeze(-1)

    if y_true.ndim != 2 or y_pred.ndim != 2:
        raise ValueError(
            f"Se espera shape (n_samples, seq_len) o (n_samples, seq_len, 1). "
            f"Recibido y_true.ndim={y_true.ndim}, y_pred.ndim={y_pred.ndim}"
        )

    if y_true.shape != y_pred.shape:
        raise ValueError(f"y_true y y_pred deben tener el mismo shape. "
                         f"Recibido y_true={y_true.shape}, y_pred={y_pred.shape}")

    n_samples, seq_len = y_true.shape
    if n_samples == 0 or seq_len == 0:
        raise ValueError("y_true/y_pred no pueden estar vacíos.")

    # 3) Validación numérica básica
    if not (np.isfinite(y_true).all() and np.isfinite(y_pred).all()):
        raise ValueError("Se encontraron NaN o inf en y_true/y_pred. "
                         "Limpie o enmascare antes de calcular métricas.")

    # 4) Errores
    errors = y_pred - y_true
    abs_errors = np.abs(errors)

    # 5) Métricas globales
    mae = abs_errors.mean()
    rmse = np.sqrt((errors ** 2).mean())

    # 6) Métrica direccional en último paso (DA_last)
    y_true_last = y_true[:, -1]
    y_pred_last = y_pred[:, -1]

    sign_true = np.sign(y_true_last)
    sign_pred = np.sign(y_pred_last)

    if da_ignore_zeros:
        mask = (sign_true != 0) & (sign_pred != 0)
        da_last = float(np.mean(sign_true[mask] == sign_pred[mask])) if mask.any() else float("nan")
        da_last_n = int(mask.sum())
    else:
        da_last = float(np.mean(sign_true == sign_pred))
        da_last_n = int(n_samples)

    # 7) MAE por paso (diagnóstico)
    mae_per_step = abs_errors.mean(axis=0)  # (seq_len,)

    metrics = {
        "MAE": float(mae),
        "RMSE": float(rmse),
        "DA_last": da_last,
        "DA_last_n": da_last_n,          # cuántas muestras realmente aportaron a DA_last
        "MAE_per_step": mae_per_step.tolist(),
    }

    # 8) R² opcional
    if compute_r2:
        metrics["R2"] = float(r2_score(y_true.ravel(), y_pred.ravel()))

    return metrics


In [ ]:
#Ejemplo de uso:
metrics = compute_seq2seq_metrics(y_true, y_pred)
print(metrics["MAE"], metrics["RMSE"], metrics["DA_last"])

## **7. Arquitectura de notebooks**

# Stage_07_XX – <MODEL_NAME> (Seq2Seq 29x8 → 29x1)

## 0) Objetivo de la notebook
- Qué modelo se entrena y por qué.
- Qué horizonte aplica (h=60 o h=90).
- Qué artefactos produce para Stage_08.

---

## 1) Setup
### 1.1 Imports
- numpy, pandas, torch/keras (según modelo)
- utilidades comunes: `compute_seq2seq_metrics`, `compute_opportunity_filter_metrics`

### 1.2 Reproducibilidad
- seeds (numpy / torch / random)
- device (cpu/cuda)
- flags de determinismo si aplica

---

## 2) Configuración (parámetros)
- `HORIZON = 60 | 90`
- `SEQ_LEN = 29`
- `N_FEATURES = 8`
- hiperparámetros del modelo
- `theta` (umbral señal) y `delta_op` (umbral oportunidad)

---

## 3) Carga de datos (inputs del pipeline)
- cargar `windows_{split}_{h}.npz` (train/valid/test)
- verificar shapes y dtypes
- sanity checks (NaN, rangos, conteos)

---

## 4) Dataloaders / batching
- dataset + dataloader
- definición clara de:
  - `X: (batch, 29, 8)`
  - `Y: (batch, 29, 1)` o `(batch, 29)`

---

## 5) Definición del modelo
- arquitectura
- función de pérdida (MSE / MAE)
- optimizer
- scheduler (opcional)

---

## 6) Entrenamiento
- loop epochs
- early stopping (por valid loss)
- logging mínimo:
  - train_loss, valid_loss por epoch

---

## 7) Evaluación (OOS)
- inferencia sobre valid y test
- cálculo métricas ML:
  - MAE, RMSE, DA_last (y R2 opcional)
- cálculo métricas económicas (filtro):
  - Precision, Opportunity_Recall, Coverage

---

## 8) Guardado de artefactos (para Stage_08)
Guardar en una estructura estándar por modelo/horizonte:

- `reports/stage_07/<h>/<model_name>/metrics_ml.json`
- `reports/stage_07/<h>/<model_name>/metrics_econ.json`
- `reports/stage_07/<h>/<model_name>/pred_test.npz`
  - y_true, y_pred (test)
- `models/stage_07/<h>/<model_name>/model.*` (pesos / joblib / pt)

---

## 9) Resumen final
- tabla corta con métricas principales
- notas de entrenamiento (tiempo, convergencia, issues)